## Using Advisor Tool

### Installing Libraries

In [ ]:
%pip install anthropic python-dotenv

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
claude_model = os.getenv("CLAUDE_MODEL")

### Creating the Anthropic Client

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=anthropic_api_key)

### Calling the Messages API with Advisor Tool

In [ ]:
EXECUTOR_MODEL = claude_model
ADVISOR_MODEL = "claude-opus-5"

SYSTEM_PROMPT = """\
You are a senior software engineer working on a technical task.

You have an `advisor` tool that consults a stronger model. It sees this entire
conversation, so you do not need to restate context when calling it.

Call the advisor:
1. BEFORE committing to an architecture or approach, once you understand the task.
2. When you are stuck, or the same problem keeps recurring.
3. BEFORE you give your final answer, to sanity-check it.

Do not call the advisor for routine or mechanical steps. When it responds, treat
its guidance as expert input: apply it, but say so explicitly if you deviate and why.

Structure your final answer as: Approach, Key decisions, Risks, Next steps.
"""

In [ ]:
def solve_with_advisor(task: str, max_advisor_calls: int = 3) -> str:
    messages = [{"role": "user", "content": task}]
    tools = [{
        "type": "advisor_20260301",
        "name": "advisor",
        "model": ADVISOR_MODEL,
        "max_uses": max_advisor_calls,   # caps cost per request
    }]

    advisor_calls = 0
    while True:
        response = client.beta.messages.create(
            model=EXECUTOR_MODEL,
            max_tokens=4096,
            system=SYSTEM_PROMPT,
            tools=tools,
            messages=messages,
            betas=["advisor-tool-2026-03-01"],
        )

        advisor_calls += sum(
            1 for b in response.content
            if b.type == "server_tool_use" and getattr(b, "name", "") == "advisor"
        )

        # Server tools can pause a turn; resume by echoing the assistant content back.
        if response.stop_reason == "pause_turn":
            messages.append({"role": "assistant", "content": response.content})
            continue

        print(f"[advisor consulted {advisor_calls}x]")
        return "".join(b.text for b in response.content if b.type == "text")


if __name__ == "__main__":
    print(solve_with_advisor(
        "Design an offline-first sync layer for an Android app using Room and "
        "WorkManager. It must handle conflict resolution, flaky networks, and "
        "schema migrations across app versions."
    ))

### Calling the Messages API with Advisor Tool (2)

In [ ]:
response = client.beta.messages.create(
    model=EXECUTOR_MODEL,
    max_tokens=4096,
    betas=["advisor-tool-2026-03-01"],
    tools=[
        {
            "type": "advisor_20260301",
            "name": "advisor",
            "model": ADVISOR_MODEL,
        }
    ],
    messages=[
        {
            "role": "user",
            "content": "Send core architecture and tech stack recommendations for client-side RAG implementation in an Android mobile app.",
        }
    ],
)

print(response.to_json())